# LeetCode 743 — Network Delay Time bằng Python

Link đề: https://leetcode.com/problems/network-delay-time/ · Độ khó: **Medium** · Chủ đề: Graph, Shortest Path, Dijkstra, Heap (Priority Queue), Breadth-First Search, Depth-First Search

Skeleton LeetCode:

```python
class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        ...
```

Ghi chú: đây là bài "đường đi ngắn nhất từ một nguồn" **kinh điển** — cửa ngõ để học **Dijkstra**. Bài chỉ hỏi *thời gian tín hiệu tới nút xa nhất* nên đáp án là **giá trị lớn nhất** trong các khoảng cách ngắn nhất, và `-1` nếu có nút không tới được. Notebook trình bày 5 cách: Bellman–Ford, Floyd–Warshall, Dijkstra `O(n^2)` (mảng), Dijkstra với heap (cách nộp), và SPFA (Bellman–Ford dùng hàng đợi).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mạng có `n` nút (đánh số `1..n`) với các cạnh **có hướng, có trọng số** `(u, v, w)`; gửi tín hiệu từ nút `k`; tìm thời gian ít nhất để **cả `n` nút** nhận được tín hiệu, hoặc `-1` nếu không thể.
2. Nắm công cụ Python cần dùng: danh sách kề, `float("inf")`, `heapq` (`heappush`, `heappop`) với phần tử là tuple, `deque`, `max` trên list, và cách **nới lỏng cạnh** (relaxation).
3. Đi từ **Bellman–Ford `O(n·E)`** → **Floyd–Warshall `O(n^3)`** → **Dijkstra `O(n^2)`** → **Dijkstra + heap `O(E log n)`** → **SPFA**.
4. Hiểu mấu chốt: đáp án = `max` của các khoảng cách ngắn nhất từ `k` (**không phải tổng**), và nếu còn `inf` thì trả `-1`.
5. Tránh các lỗi kinh điển: lấy tổng thay vì `max`, quên nút không tới được, nhầm chỉ số `1..n` với mảng `0..n-1`, đánh dấu "đã thăm" lúc **đẩy vào** heap thay vì lúc **lấy ra**, nhầm với BFS không trọng số, và coi cạnh là vô hướng.

## 2. Đề bài & ví dụ

Cho một mạng gồm `n` nút, đánh số từ `1` đến `n`. Cho thêm danh sách `times`, mỗi phần tử `times[i] = (u, v, w)` là một **cạnh có hướng**: tín hiệu đi từ nút nguồn `u` tới nút đích `v` mất `w` đơn vị thời gian.

Ta gửi một tín hiệu từ nút `k`. Hãy trả về **thời gian tối thiểu** để **tất cả `n` nút** nhận được tín hiệu. Nếu không thể để cả `n` nút đều nhận được, trả về `-1`.

**Ví dụ 1**

- Input: `times = [[2,1,1],[2,3,1],[3,4,1]]`, `n = 4`, `k = 2`
- Output: `2`
- Giải thích: từ `2` tới `1` mất 1, tới `3` mất 1, từ `3` tới `4` mất thêm 1 (tổng 2). Nút xa nhất nhận lúc thời điểm `2`.

**Ví dụ 2**

- Input: `times = [[1,2,1]]`, `n = 2`, `k = 1`
- Output: `1`

**Ví dụ 3**

- Input: `times = [[1,2,1]]`, `n = 2`, `k = 2`
- Output: `-1`
- Giải thích: cạnh chỉ đi từ `1` tới `2`; từ `2` không tới được `1`.

**Ràng buộc:**

- `1 <= k <= n <= 100`
- `1 <= times.length <= 6000`
- `times[i].length == 3`, `1 <= u, v <= n`, `u != v`
- `0 <= w <= 100` (trọng số **không âm**, có thể bằng `0`)
- Mọi cặp `(u, v)` đều **khác nhau** (không có cạnh bội).

Nhận xét: `n <= 100` rất nhỏ nên các cách `O(n^3)` (Floyd–Warshall) vẫn chạy được; nhưng bài này là dịp học thuật toán chuẩn là **Dijkstra**.

In [1]:
from typing import List
from collections import deque
from heapq import heappush, heappop
import random

inf = float("inf")

examples = [
    ([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2, 2),
    ([[1, 2, 1]], 2, 1, 1),
    ([[1, 2, 1]], 2, 2, -1),
]
for times, n, k, expected in examples:
    print("times =", times, "| n =", n, "| k =", k, "| kỳ vọng:", expected)

times = [[2, 1, 1], [2, 3, 1], [3, 4, 1]] | n = 4 | k = 2 | kỳ vọng: 2
times = [[1, 2, 1]] | n = 2 | k = 1 | kỳ vọng: 1
times = [[1, 2, 1]] | n = 2 | k = 2 | kỳ vọng: -1


## 3. Kiến thức nền

### 3.1. Đồ thị có hướng, có trọng số và danh sách kề

Mỗi nút là một "máy"; mỗi cạnh `(u, v, w)` là một đường truyền một chiều từ `u` tới `v` tốn `w`. Để duyệt hiệu quả ta lưu **danh sách kề**: `g[u]` là danh sách các cặp `(v, w)` đi ra từ `u`. Vì nút đánh số `1..n`, ta cho `g` có `n + 1` phần tử và bỏ phần tử `0` — tiện hơn là trừ 1 mỗi lần (ta sẽ thấy lỗi chỉ số ở mục 13).

In [2]:
def build_graph(times, n):
    g = [[] for _ in range(n + 1)]          # g[0] không dùng
    for u, v, w in times:
        g[u].append((v, w))                  # cạnh CÓ HƯỚNG u -> v
    return g


g = build_graph(examples[0][0], 4)
for u in range(1, 5):
    print(f"g[{u}] =", g[u])

g[1] = []
g[2] = [(1, 1), (3, 1)]
g[3] = [(4, 1)]
g[4] = []


### 3.2. `float("inf")` — "chưa biết khoảng cách"

`inf` là số dương lớn hơn mọi số khác; `inf + w = inf`; `min(inf, 5) = 5`. Ta khởi tạo `dist[x] = inf` cho mọi nút (chưa tới được) và `dist[k] = 0`. Cuối cùng nếu `max(dist)` vẫn là `inf` thì có nút không tới được → `-1`.

In [3]:
print(inf > 10 ** 18, "| inf + 5 =", inf + 5, "| min(inf, 5) =", min(inf, 5), "| inf == inf:", inf == inf)
dist = [inf, 0, inf, 3]
print("max =", max(dist), "| max có bằng inf?", max(dist) == inf)

True | inf + 5 = inf | min(inf, 5) = 5 | inf == inf: True
max = inf | max có bằng inf? True


### 3.3. Nới lỏng cạnh (relaxation)

Với cạnh `(u, v, w)`: nếu đi tới `u` mất `dist[u]` thì đi tiếp sang `v` qua cạnh này mất `dist[u] + w`. Nếu con số đó **nhỏ hơn** `dist[v]` hiện có thì ta cập nhật: `dist[v] = dist[u] + w`. Thao tác này gọi là **relaxation** và là "viên gạch" của mọi thuật toán đường ngắn nhất.

In [4]:
dist = [inf] * 5
dist[2] = 0                                  # nguồn k = 2
for u, v, w in [(2, 1, 1), (2, 3, 1), (3, 4, 1)]:
    if dist[u] + w < dist[v]:
        print(f"nới lỏng ({u},{v},{w}): dist[{v}] {dist[v]} -> {dist[u] + w}")
        dist[v] = dist[u] + w
    else:
        print(f"nới lỏng ({u},{v},{w}): không cải thiện (dist[{u}] = {dist[u]})")
print(dist)

nới lỏng (2,1,1): dist[1] inf -> 1
nới lỏng (2,3,1): dist[3] inf -> 1
nới lỏng (3,4,1): dist[4] inf -> 2
[inf, 1, 0, 1, 2]


Lần duyệt cạnh `(3, 4, 1)` ở trên đứng **sau** `(2, 3, 1)` nên mới cập nhật được; nếu thứ tự đảo lại thì vòng đầu chưa cải thiện — vì vậy các thuật toán như Bellman–Ford phải lặp nhiều vòng.

### 3.4. `heapq`: hàng đợi ưu tiên với tuple

`heapq` là **min-heap** trên list: `heappush(h, x)` thêm `x`; `heappop(h)` lấy **phần tử nhỏ nhất** ra, cả hai tốn `O(log n)`. Với tuple, Python so sánh **từng phần tử từ trái sang phải**, nên `(khoảng cách, nút)` sẽ xếp theo khoảng cách nhỏ nhất trước (và nếu bằng thì theo số hiệu nút).

In [5]:
h = []
for item in [(5, 2), (1, 9), (3, 1), (1, 4)]:
    heappush(h, item)
print("lấy ra theo thứ tự:", [heappop(h) for _ in range(len(h))])

lấy ra theo thứ tự: [(1, 4), (1, 9), (3, 1), (5, 2)]


### 3.5. Vì sao Dijkstra cần trọng số **không âm**?

Ý tưởng Dijkstra: luôn "chốt" nút có khoảng cách **tạm thời nhỏ nhất** chưa chốt. Nếu mọi `w >= 0` thì mọi đường đi khác tới nút đó phải đi qua một nút chưa chốt có khoảng cách tạm **không nhỏ hơn**, và cộng thêm các cạnh không âm sẽ **không giảm** — nên khoảng cách đã chốt là cuối cùng. Với trọng số âm, lập luận này sụp đổ (khi đó dùng Bellman–Ford). Đề bài bảo đảm `w >= 0` (kể cả `0`).

## 4. Ý tưởng nền tảng / trực giác

Tín hiệu đi **song song** theo mọi cạnh; nút `x` nhận được tín hiệu vào thời điểm `dist[x]` = **độ dài đường ngắn nhất** từ `k` tới `x`. Cả `n` nút đã nhận xong khi nút **nhận muộn nhất** nhận được, tức:

> đáp án = `max` của `dist[x]` trên mọi nút `x`; nếu có `dist[x] = inf` thì `-1`.

Tính tay ví dụ 1 (`k = 2`):

| nút | đường ngắn nhất từ `2` | `dist` |
|---|---|---|
| 2 | (chính nó) | 0 |
| 1 | `2 → 1` | 1 |
| 3 | `2 → 3` | 1 |
| 4 | `2 → 3 → 4` | 2 |

`max = 2`. Ví dụ 3: `k = 2` không có cạnh đi ra → `dist[1] = inf` → `-1`.

Lưu ý: đáp án **không phải tổng** các thời gian — các nhánh chạy song song, ta chỉ chờ nhánh chậm nhất.

In [6]:
def show_dist(times, n, k):
    """Tính dist bằng cách rất đơn giản (lặp relaxation đến khi hết thay đổi) để quan sát."""
    dist = [inf] * (n + 1)
    dist[k] = 0
    changed = True
    while changed:
        changed = False
        for u, v, w in times:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                changed = True
    return dist[1:]


for times, n, k, expected in examples:
    d = show_dist(times, n, k)
    print(f"k = {k}: dist (nút 1..n) = {d} -> max = {max(d)} ->", "-1" if max(d) == inf else max(d), "| kỳ vọng", expected)

k = 2: dist (nút 1..n) = [1, 0, 1, 2] -> max = 2 -> 2 | kỳ vọng 2
k = 1: dist (nút 1..n) = [0, 1] -> max = 1 -> 1 | kỳ vọng 1
k = 2: dist (nút 1..n) = [inf, 0] -> max = inf -> -1 | kỳ vọng -1


## 5. Cách 1 — Brute Force / baseline: Bellman–Ford

### Ý tưởng

Đường đi ngắn nhất **không có chu trình** (vì trọng số không âm) nên có tối đa `n - 1` cạnh. Do đó nếu ta lặp `n - 1` vòng, mỗi vòng **nới lỏng mọi cạnh**, thì sau vòng thứ `i` mọi nút có đường ngắn nhất gồm `<= i` cạnh đã đúng. Dừng sớm nếu một vòng không thay đổi gì.

Ta cũng viết thêm một **oracle** thật ngây thơ: liệt kê **mọi đường đi đơn** từ `k` bằng DFS (cho đồ thị nhỏ) và lấy thời gian nhỏ nhất tới từng nút. Oracle này không dùng bất kỳ thuật toán đường ngắn nhất nào nên rất đáng tin để đối chiếu.

In [7]:
def delay_bellman_ford(times, n, k):
    dist = [inf] * (n + 1)
    dist[k] = 0
    for _ in range(n - 1):
        changed = False
        for u, v, w in times:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                changed = True
        if not changed:
            break
    ans = max(dist[1:])
    return -1 if ans == inf else ans


def oracle_dists(times, n, k, max_edges=None):
    """Oracle: DFS mọi đường đơn từ k (chỉ dùng cho đồ thị nhỏ). max_edges giới hạn số cạnh."""
    g = build_graph(times, n)
    best = [inf] * (n + 1)
    best[k] = 0

    def dfs(u, t, used, visited):
        if max_edges is not None and used == max_edges:
            return
        for v, w in g[u]:
            if v in visited:
                continue
            if t + w < best[v]:
                best[v] = t + w
            dfs(v, t + w, used + 1, visited | {v})

    dfs(k, 0, 0, {k})
    return best


def oracle_delay(times, n, k):
    ans = max(oracle_dists(times, n, k)[1:])
    return -1 if ans == inf else ans


for times, n, k, expected in examples:
    print(times, k, "| Bellman-Ford:", delay_bellman_ford(times, n, k), "| oracle:", oracle_delay(times, n, k), "| kỳ vọng:", expected)
    assert delay_bellman_ford(times, n, k) == oracle_delay(times, n, k) == expected

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] 2 | Bellman-Ford: 2 | oracle: 2 | kỳ vọng: 2
[[1, 2, 1]] 1 | Bellman-Ford: 1 | oracle: 1 | kỳ vọng: 1
[[1, 2, 1]] 2 | Bellman-Ford: -1 | oracle: -1 | kỳ vọng: -1


### Độ phức tạp và nút thắt

- Tối đa `n - 1` vòng, mỗi vòng duyệt `E` cạnh: `O(n · E)` thời gian. Với `n = 100`, `E = 6000`: khoảng `6 * 10^5` — **ổn** với bài này.
- Bộ nhớ `O(n)`.

Nút thắt: ta nới lỏng cả những cạnh **chưa có ý nghĩa** (`dist[u]` còn `inf`) và quét lại **toàn bộ** cạnh mỗi vòng. Dijkstra làm khéo hơn: mỗi lần chỉ xử lý nút có khoảng cách nhỏ nhất, và mỗi nút chỉ mở rộng **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Dijkstra:** duy trì tập nút đã **chốt** (khoảng cách cuối cùng). Mỗi bước:

1. Chọn nút `u` **chưa chốt** có `dist[u]` **nhỏ nhất**; chốt `u`.
2. Nới lỏng mọi cạnh đi ra từ `u`: với `(u, v, w)` cập nhật `dist[v] = min(dist[v], dist[u] + w)`.

Vì mọi `w >= 0`, khoảng cách của `u` lúc chốt là cuối cùng (mục 3.5). Mỗi nút chốt đúng một lần, mỗi cạnh nới lỏng đúng một lần.

Có hai cách "chọn nút nhỏ nhất":

- **Quét mảng** `O(n)` mỗi lần → tổng `O(n^2)` (cách 3).
- **Min-heap** `O(log n)` mỗi lần → tổng `O((n + E) log n)` (cách 4).

Đáp án cuối: `max(dist)` (hoặc `-1` nếu còn `inf`).

## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) Bellman–Ford (mục 5), (2) Floyd–Warshall, (3) Dijkstra mảng `O(n^2)`, (4) Dijkstra + heap, (5) SPFA.

### 7.1. Cách 2 — Floyd–Warshall `O(n^3)`

**Ý tưởng.** Tính khoảng cách ngắn nhất giữa **mọi cặp** nút bằng quy hoạch động: `d[i][j]` ban đầu là trọng số cạnh trực tiếp (hoặc `inf`; `d[i][i] = 0`). Với mỗi nút trung gian `t` lần lượt `1..n`, thử cải thiện `d[i][j] = min(d[i][j], d[i][t] + d[t][j])` cho mọi `i, j`. Sau khi dùng hết các nút trung gian, `d[i][j]` là đường ngắn nhất. Ta chỉ cần hàng `k`.

Với `n = 100`, `n^3 = 10^6` — vừa đủ cho bài này, nhưng làm thừa vì ta chỉ cần **một nguồn**.

In [8]:
def delay_floyd(times, n, k):
    d = [[inf] * (n + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        d[i][i] = 0
    for u, v, w in times:
        d[u][v] = min(d[u][v], w)
    for t in range(1, n + 1):
        dt = d[t]
        for i in range(1, n + 1):
            dit = d[i][t]
            if dit == inf:
                continue
            di = d[i]
            for j in range(1, n + 1):
                if dit + dt[j] < di[j]:
                    di[j] = dit + dt[j]
    ans = max(d[k][1:])
    return -1 if ans == inf else ans


for times, n, k, expected in examples:
    print(times, k, "->", delay_floyd(times, n, k))
    assert delay_floyd(times, n, k) == expected

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] 2 -> 2
[[1, 2, 1]] 1 -> 1
[[1, 2, 1]] 2 -> -1


**Code nộp LeetCode (cách 2):**

```python
class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        d = [[inf] * (n + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            d[i][i] = 0
        for u, v, w in times:
            d[u][v] = min(d[u][v], w)
        for t in range(1, n + 1):
            for i in range(1, n + 1):
                for j in range(1, n + 1):
                    if d[i][t] + d[t][j] < d[i][j]:
                        d[i][j] = d[i][t] + d[t][j]
        ans = max(d[k][1:])
        return -1 if ans == inf else ans
```

**Độ phức tạp:** Time `O(n^3)`, Space `O(n^2)`.

### 7.2. Cách 3 — Dijkstra `O(n^2)` bằng quét mảng

**Giải thích từng đoạn.**

1. Ma trận kề `w[u][v]` (hoặc `inf` nếu không có cạnh), `dist = [inf]*(n+1)`, `dist[k] = 0`, `done = [False]*(n+1)`.
2. Lặp `n` lần: tìm nút `u` chưa `done` có `dist[u]` nhỏ nhất (quét `O(n)`); nếu `dist[u]` là `inf` thì dừng (không còn nút tới được).
3. Đánh dấu `done[u] = True`; với mọi `v`: `dist[v] = min(dist[v], dist[u] + w[u][v])`.

In [9]:
def delay_dijkstra_array(times, n, k):
    w = [[inf] * (n + 1) for _ in range(n + 1)]
    for u, v, t in times:
        w[u][v] = t
    dist = [inf] * (n + 1)
    dist[k] = 0
    done = [False] * (n + 1)
    for _ in range(n):
        u = -1
        for x in range(1, n + 1):
            if not done[x] and (u == -1 or dist[x] < dist[u]):
                u = x
        if dist[u] == inf:
            break
        done[u] = True
        for v in range(1, n + 1):
            if dist[u] + w[u][v] < dist[v]:
                dist[v] = dist[u] + w[u][v]
    ans = max(dist[1:])
    return -1 if ans == inf else ans


for times, n, k, expected in examples:
    print(times, k, "->", delay_dijkstra_array(times, n, k))
    assert delay_dijkstra_array(times, n, k) == expected

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] 2 -> 2
[[1, 2, 1]] 1 -> 1
[[1, 2, 1]] 2 -> -1


**Code nộp LeetCode (cách 3):**

```python
class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        w = [[inf] * (n + 1) for _ in range(n + 1)]
        for u, v, t in times:
            w[u][v] = t
        dist = [inf] * (n + 1)
        dist[k] = 0
        done = [False] * (n + 1)
        for _ in range(n):
            u = -1
            for x in range(1, n + 1):
                if not done[x] and (u == -1 or dist[x] < dist[u]):
                    u = x
            if dist[u] == inf:
                break
            done[u] = True
            for v in range(1, n + 1):
                if dist[u] + w[u][v] < dist[v]:
                    dist[v] = dist[u] + w[u][v]
        ans = max(dist[1:])
        return -1 if ans == inf else ans
```

**Độ phức tạp:** Time `O(n^2)` (cộng `O(E)` dựng ma trận), Space `O(n^2)` cho ma trận kề. Cách này hợp với đồ thị **dày**.

### 7.3. Cách 4 — Dijkstra với min-heap (cách nộp)

**Giải thích từng đoạn.**

1. `g = build_graph(times, n)`; `dist = [inf]*(n+1)`; `dist[k] = 0`; `heap = [(0, k)]`.
2. Lặp khi `heap` còn phần tử: `d, u = heappop(heap)`.
3. Nếu `d > dist[u]` thì đây là **bản ghi cũ** (stale, đã có đường tốt hơn) → bỏ qua. Đây là cách thay cho thao tác "giảm khóa" mà `heapq` không có.
4. Với mỗi `(v, w)` trong `g[u]`: nếu `d + w < dist[v]` thì cập nhật `dist[v]` và `heappush(heap, (d + w, v))`.
5. Trả `max(dist[1:])` hoặc `-1` nếu còn `inf`.

In [10]:
def delay_dijkstra_heap(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:                    # bản ghi cũ: bỏ qua
            continue
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heappush(heap, (nd, v))
    ans = max(dist[1:])
    return -1 if ans == inf else ans


for times, n, k, expected in examples:
    print(times, k, "->", delay_dijkstra_heap(times, n, k))
    assert delay_dijkstra_heap(times, n, k) == expected

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] 2 -> 2
[[1, 2, 1]] 1 -> 1
[[1, 2, 1]] 2 -> -1


**Code nộp LeetCode (cách 4):**

```python
from heapq import heappush, heappop

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        g = [[] for _ in range(n + 1)]
        for u, v, w in times:
            g[u].append((v, w))
        dist = [inf] * (n + 1)
        dist[k] = 0
        heap = [(0, k)]
        while heap:
            d, u = heappop(heap)
            if d > dist[u]:
                continue
            for v, w in g[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    heappush(heap, (nd, v))
        ans = max(dist[1:])
        return -1 if ans == inf else ans
```

**Độ phức tạp:** Time `O((n + E) log E)` ≈ `O(E log n)`, Space `O(n + E)`. Mỗi cạnh có thể đẩy tối đa một bản ghi vào heap.

### 7.4. Cách 5 — SPFA (Bellman–Ford dùng hàng đợi)

**Ý tưởng.** Bellman–Ford chỉ cần nới lỏng các cạnh **đi ra từ nút vừa được cải thiện**. SPFA (Shortest Path Faster Algorithm) giữ một hàng đợi các nút vừa cập nhật `dist`; mỗi lần lấy nút `u` ra, nới lỏng các cạnh của `u`; nếu `dist[v]` giảm và `v` chưa trong hàng đợi thì đưa `v` vào. Trung bình nhanh, nhưng xấu nhất vẫn `O(n·E)`.

In [11]:
def delay_spfa(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    q = deque([k])
    in_queue = [False] * (n + 1)
    in_queue[k] = True
    while q:
        u = q.popleft()
        in_queue[u] = False
        for v, w in g[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                if not in_queue[v]:
                    in_queue[v] = True
                    q.append(v)
    ans = max(dist[1:])
    return -1 if ans == inf else ans


for times, n, k, expected in examples:
    print(times, k, "->", delay_spfa(times, n, k))
    assert delay_spfa(times, n, k) == expected

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] 2 -> 2
[[1, 2, 1]] 1 -> 1
[[1, 2, 1]] 2 -> -1


**Code nộp LeetCode (cách 5):**

```python
from collections import deque

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        g = [[] for _ in range(n + 1)]
        for u, v, w in times:
            g[u].append((v, w))
        dist = [inf] * (n + 1)
        dist[k] = 0
        q = deque([k])
        in_queue = [False] * (n + 1)
        in_queue[k] = True
        while q:
            u = q.popleft()
            in_queue[u] = False
            for v, w in g[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    if not in_queue[v]:
                        in_queue[v] = True
                        q.append(v)
        ans = max(dist[1:])
        return -1 if ans == inf else ans
```

**Độ phức tạp:** Time trung bình `O(E)`–`O(E log n)`, xấu nhất `O(n·E)`; Space `O(n + E)`. Với trọng số không âm và `n <= 100` mọi trường hợp đều nhỏ.

Năm cách cho cùng đáp án; mục 12 sẽ đối chiếu bằng `assert` với `oracle_delay` (DFS mọi đường đơn) và với nhau trên đồ thị lớn.

## 8. TRACE từng bước

Trace Dijkstra heap (cách 4). Bảng tay cho ví dụ 1 (`k = 2`, cạnh `2→1 (1)`, `2→3 (1)`, `3→4 (1)`):

| bước | lấy ra `(d, u)` | bản ghi cũ? | nới lỏng | `dist[1..4]` | heap sau |
|---|---|---|---|---|---|
| bắt đầu | - | - | - | `[inf, 0, inf, inf]` | `[(0,2)]` |
| 1 | `(0, 2)` | không | `1: inf→1`, `3: inf→1` | `[1, 0, 1, inf]` | `[(1,1), (1,3)]` |
| 2 | `(1, 1)` | không | `1` không có cạnh đi ra | `[1, 0, 1, inf]` | `[(1,3)]` |
| 3 | `(1, 3)` | không | `4: inf→2` | `[1, 0, 1, 2]` | `[(2,4)]` |
| 4 | `(2, 4)` | không | không có cạnh đi ra | `[1, 0, 1, 2]` | `[]` |

`max(dist) = 2`.

In [12]:
def trace_dijkstra(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    print(f"bắt đầu: dist = {dist[1:]}, heap = {heap}")
    step = 0
    while heap:
        d, u = heappop(heap)
        step += 1
        if d > dist[u]:
            print(f"bước {step}: lấy ({d},{u}) -> BẢN GHI CŨ (dist[{u}] = {dist[u]}), bỏ qua")
            continue
        notes = []
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                notes.append(f"{v}: {dist[v]}->{nd}")
                dist[v] = nd
                heappush(heap, (nd, v))
            else:
                notes.append(f"{v}: giữ {dist[v]} (đường qua {u} = {nd})")
        print(f"bước {step}: lấy ({d},{u}) chốt nút {u}; nới lỏng [{'; '.join(notes) or 'không có cạnh đi ra'}] "
              f"| dist = {dist[1:]} | heap = {sorted(heap)}")
    ans = max(dist[1:])
    print("max(dist) =", ans, "->", "-1" if ans == inf else ans)
    return -1 if ans == inf else ans


assert trace_dijkstra([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2

bắt đầu: dist = [inf, 0, inf, inf], heap = [(0, 2)]
bước 1: lấy (0,2) chốt nút 2; nới lỏng [1: inf->1; 3: inf->1] | dist = [1, 0, 1, inf] | heap = [(1, 1), (1, 3)]
bước 2: lấy (1,1) chốt nút 1; nới lỏng [không có cạnh đi ra] | dist = [1, 0, 1, inf] | heap = [(1, 3)]
bước 3: lấy (1,3) chốt nút 3; nới lỏng [4: inf->2] | dist = [1, 0, 1, 2] | heap = [(2, 4)]
bước 4: lấy (2,4) chốt nút 4; nới lỏng [không có cạnh đi ra] | dist = [1, 0, 1, 2] | heap = []
max(dist) = 2 -> 2


Trace thêm một đồ thị có **đường tắt** và **bản ghi cũ**: `1→2 (5)`, `1→3 (1)`, `3→2 (1)`, `2→4 (1)`, `k = 1`. Nút `2` đầu tiên được ghi với `5` rồi được cải thiện thành `2` qua nút `3`; bản ghi `(5, 2)` sẽ trở thành "cũ" khi lấy ra.

In [13]:
res = trace_dijkstra([[1, 2, 5], [1, 3, 1], [3, 2, 1], [2, 4, 1]], 4, 1)
assert res == 3 == oracle_delay([[1, 2, 5], [1, 3, 1], [3, 2, 1], [2, 4, 1]], 4, 1)

bắt đầu: dist = [0, inf, inf, inf], heap = [(0, 1)]
bước 1: lấy (0,1) chốt nút 1; nới lỏng [2: inf->5; 3: inf->1] | dist = [0, 5, 1, inf] | heap = [(1, 3), (5, 2)]
bước 2: lấy (1,3) chốt nút 3; nới lỏng [2: 5->2] | dist = [0, 2, 1, inf] | heap = [(2, 2), (5, 2)]
bước 3: lấy (2,2) chốt nút 2; nới lỏng [4: inf->3] | dist = [0, 2, 1, 3] | heap = [(3, 4), (5, 2)]
bước 4: lấy (3,4) chốt nút 4; nới lỏng [không có cạnh đi ra] | dist = [0, 2, 1, 3] | heap = [(5, 2)]
bước 5: lấy (5,2) -> BẢN GHI CŨ (dist[2] = 2), bỏ qua
max(dist) = 3 -> 3


Cuối cùng trace ví dụ 3 (`k = 2`, chỉ có cạnh `1→2`): nút `1` không bao giờ được chạm tới nên còn `inf` → `-1`.

In [14]:
assert trace_dijkstra([[1, 2, 1]], 2, 2) == -1

bắt đầu: dist = [inf, 0], heap = [(0, 2)]
bước 1: lấy (0,2) chốt nút 2; nới lỏng [không có cạnh đi ra] | dist = [inf, 0] | heap = []
max(dist) = inf -> -1


## 9. Code nộp LeetCode

Đây là cách 4 (Dijkstra + heap), cách chuẩn nhất cho dạng bài này.

```python
from heapq import heappush, heappop
from typing import List

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        g = [[] for _ in range(n + 1)]            # g[0] không dùng, nút đánh số 1..n
        for u, v, w in times:
            g[u].append((v, w))
        dist = [inf] * (n + 1)
        dist[k] = 0
        heap = [(0, k)]
        while heap:
            d, u = heappop(heap)
            if d > dist[u]:                       # bản ghi cũ
                continue
            for v, w in g[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    heappush(heap, (nd, v))
        ans = max(dist[1:])                       # bỏ phần tử 0
        return -1 if ans == inf else ans
```

In [15]:
class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        g = [[] for _ in range(n + 1)]
        for u, v, w in times:
            g[u].append((v, w))
        dist = [inf] * (n + 1)
        dist[k] = 0
        heap = [(0, k)]
        while heap:
            d, u = heappop(heap)
            if d > dist[u]:
                continue
            for v, w in g[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    heappush(heap, (nd, v))
        ans = max(dist[1:])
        return -1 if ans == inf else ans


sol = Solution()
for times, n, k, expected in examples:
    got = sol.networkDelayTime(times, n, k)
    print(times, k, "->", got, "| kỳ vọng", expected)
    assert got == expected

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] 2 -> 2 | kỳ vọng 2
[[1, 2, 1]] 1 -> 1 | kỳ vọng 1
[[1, 2, 1]] 2 -> -1 | kỳ vọng -1


## 10. Vì sao đúng

**Phát biểu.** Khi nút `u` được lấy ra khỏi heap với `d == dist[u]` (bản ghi hợp lệ đầu tiên của `u`), `d` là **khoảng cách ngắn nhất thật sự** từ `k` tới `u`.

**Chứng minh ngắn (quy nạp theo thứ tự chốt).**

1. Nút đầu tiên được chốt là `k` với `0` — đúng.
2. Giả sử mọi nút đã chốt trước `u` có khoảng cách đúng. Xét đường ngắn nhất thật `P` từ `k` tới `u`. Gọi `x` là nút **đầu tiên trên `P` chưa được chốt** và `y` là nút đứng ngay trước nó (đã chốt). Khi `y` được chốt, cạnh `(y, x)` đã được nới lỏng nên `dist[x] <= dist[y] + w(y, x) = độ dài đoạn P tới x`.
3. Vì mọi trọng số không âm, độ dài đoạn của `P` tới `x` là `<=` độ dài cả `P` = khoảng cách thật `d*(u)`. Vậy `dist[x] <= d*(u)`.
4. `u` được chọn vì `dist[u]` là **nhỏ nhất** trong các nút chưa chốt, nên `dist[u] <= dist[x] <= d*(u)`. Mà `dist[u] >= d*(u)` (nó luôn là độ dài một đường đi có thật). Suy ra `dist[u] = d*(u)`.
5. Bản ghi cũ bị bỏ qua nhờ `d > dist[u]`; chỉ có bản ghi hợp lệ nhỏ nhất của mỗi nút được mở rộng, nên mỗi nút được mở rộng (chốt) đúng một lần.
6. Nút không bao giờ được chạm tới giữ `inf` — chính xác là các nút không tới được từ `k`; khi đó trả `-1`.

Đáp án là `max` của khoảng cách ngắn nhất vì nút `x` nhận tín hiệu ở thời điểm `dist[x]` (mọi nhánh chạy song song) và cả mạng nhận xong khi nút muộn nhất nhận xong.

**Với Bellman–Ford:** sau `i` vòng mọi nút có đường ngắn nhất `<= i` cạnh đã đúng; đường ngắn nhất không có chu trình (vì `w >= 0`) nên `<= n - 1` cạnh. **Với Floyd–Warshall:** quy nạp theo tập nút trung gian cho phép `{1..t}`. **Với SPFA:** là Bellman–Ford chỉ nới lỏng những cạnh có thể thay đổi.

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| oracle (DFS mọi đường đơn) | mũ | `O(n)` | chỉ cho đồ thị rất nhỏ |
| 1. Bellman–Ford | `O(n·E)` | `O(n)` | đơn giản, chịu được trọng số âm |
| 2. Floyd–Warshall | `O(n^3)` | `O(n^2)` | mọi cặp, thừa cho một nguồn |
| 3. Dijkstra mảng | `O(n^2)` | `O(n^2)` | tốt cho đồ thị dày, `n` nhỏ |
| 4. Dijkstra + heap (`Solution`) | `O(E log n)` | `O(n + E)` | cách nộp, đồ thị thưa |
| 5. SPFA | `O(E)` trung bình, `O(n·E)` xấu nhất | `O(n + E)` | nhanh thực tế, không đảm bảo xấu nhất |

Với `n <= 100` và `E <= 6000`: Dijkstra heap cỡ `6000 * log` thao tác; Bellman–Ford cỡ `6 * 10^5`; Floyd cỡ `10^6`. Tất cả đều dưới một giây trong Python, nhưng Dijkstra heap là tiêu chuẩn mở rộng được.

## 12. Bộ test

Chiến lược: (a) ví dụ của đề, (b) edge case (nút không tới được, trọng số `0`, chu trình, đường tắt, đồ thị chuỗi dài), (c) phản ví dụ cho các lời giải sai, (d) đối chiếu **mọi cách** với `oracle_delay` (DFS mọi đường đơn) trên đồ thị nhỏ ngẫu nhiên có seed, (e) đối chiếu các cách với nhau trên đồ thị **đúng cỡ ràng buộc** (`n = 100`, `E = 6000`).

### 12.1. Ví dụ của đề và edge case

In [16]:
ALL = {
    "bellman_ford": delay_bellman_ford,
    "floyd": delay_floyd,
    "dijkstra_array": delay_dijkstra_array,
    "dijkstra_heap": delay_dijkstra_heap,
    "spfa": delay_spfa,
    "solution": lambda times, n, k: Solution().networkDelayTime(times, n, k),
}

cases = [
    ([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2, 2),                      # ví dụ 1
    ([[1, 2, 1]], 2, 1, 1),                                            # ví dụ 2
    ([[1, 2, 1]], 2, 2, -1),                                           # ví dụ 3
    ([[1, 2, 1], [2, 3, 2], [3, 4, 3]], 4, 1, 6),                      # chuỗi: 1 + 2 + 3
    ([[1, 2, 0], [2, 3, 0]], 3, 1, 0),                                 # trọng số 0
    ([[1, 2, 1], [3, 2, 1]], 3, 1, -1),                                # nút 3 không tới được
    ([[1, 2, 1], [2, 1, 1], [2, 3, 1]], 3, 1, 2),                      # có chu trình
    ([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1, 2),                      # đường tắt qua 3
    ([[1, 2, 10], [1, 3, 3], [3, 2, 3], [2, 4, 1]], 4, 1, 7),          # 1->3->2->4 = 3+3+1
    ([[2, 1, 7]], 3, 1, -1),                                           # k không có cạnh đi ra
    ([[1, 2, 1], [1, 3, 1], [1, 4, 1]], 4, 1, 1),                      # hình sao
    ([[1, 2, 4], [1, 3, 1], [3, 2, 2], [2, 4, 1]], 4, 1, 4),           # dùng ở bài 2
]
# dây chuyền 100 nút, mỗi cạnh 100: đáp án 99 * 100
chain100 = [[i, i + 1, 100] for i in range(1, 100)]
cases.append((chain100, 100, 1, 9900))

for times, n, k, expected in cases:
    assert oracle_delay(times, n, k) == expected if n <= 8 else True
    for name, fn in ALL.items():
        got = fn(times, n, k)
        assert got == expected, (name, times, n, k, got, expected)
print("tất cả", len(cases), "ca cố định đều đúng với", len(ALL), "cách")

tất cả 13 ca cố định đều đúng với 6 cách


### 12.2. Phản ví dụ cho các lời giải sai

Chi tiết ở mục 13; các ca làm lộ lỗi (dùng làm hồi quy):

- Ví dụ 1 → đúng `2` (không phải tổng `4`).
- Ví dụ 3 → đúng `-1` (không phải `0`).
- `[[1,2,5],[1,3,1],[3,2,1]]`, `k = 1` → đúng `2` (đánh dấu thăm lúc đẩy vào heap cho `5`; BFS không trọng số cho `1`).

In [17]:
for times, n, k, expected in [
    ([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2, 2),
    ([[1, 2, 1]], 2, 2, -1),
    ([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1, 2),
]:
    assert oracle_delay(times, n, k) == expected
    for name, fn in ALL.items():
        assert fn(times, n, k) == expected, (name, times)
print("ca hồi quy đều đúng")

ca hồi quy đều đúng


### 12.3. Đồ thị nhỏ ngẫu nhiên so với oracle (DFS mọi đường đơn)

Đồ thị có `n` từ `2` đến `6`, số cạnh ngẫu nhiên, trọng số `0..6` (nhiều số `0` và nhiều trùng để làm khó). Cạnh được chọn **không trùng cặp** và `u != v`, đúng ràng buộc.

In [18]:
def random_graph(rng, n, m, wmax):
    pairs = [(u, v) for u in range(1, n + 1) for v in range(1, n + 1) if u != v]
    rng.shuffle(pairs)
    return [[u, v, rng.randint(0, wmax)] for u, v in pairs[:m]]


rng = random.Random(743)
stats = {"-1": 0, "ok": 0}
for _ in range(2500):
    n = rng.randint(2, 6)
    m = rng.randint(1, min(12, n * (n - 1)))
    times = random_graph(rng, n, m, 6)
    k = rng.randint(1, n)
    expected = oracle_delay(times, n, k)
    stats["-1" if expected == -1 else "ok"] += 1
    for name, fn in ALL.items():
        assert fn(times, n, k) == expected, (name, times, n, k, fn(times, n, k), expected)
print("2500 đồ thị nhỏ khớp oracle | có đáp án:", stats["ok"], "| trả -1:", stats["-1"])
assert stats["-1"] > 0 and stats["ok"] > 0

2500 đồ thị nhỏ khớp oracle | có đáp án: 1230 | trả -1: 1270


### 12.4. Đồ thị cỡ vừa: so các cách với nhau

Khi `n` lớn hơn thì oracle mũ không chạy được; ta đối chiếu năm cách (đều là thuật toán đường ngắn nhất độc lập: quét cạnh, DP trung gian, chọn nhỏ nhất bằng mảng, bằng heap, bằng hàng đợi).

In [19]:
rng = random.Random(1743)
for _ in range(150):
    n = rng.randint(5, 30)
    m = rng.randint(1, min(200, n * (n - 1)))
    times = random_graph(rng, n, m, rng.choice([1, 5, 100]))
    k = rng.randint(1, n)
    res = {name: fn(times, n, k) for name, fn in ALL.items()}
    assert len(set(res.values())) == 1, (times, n, k, res)
print("150 đồ thị cỡ vừa: 6 hàm cho cùng đáp án")

150 đồ thị cỡ vừa: 6 hàm cho cùng đáp án


### 12.5. Đồ thị đúng cỡ ràng buộc (`n = 100`, `E = 6000`)

In [20]:
import time

rng = random.Random(100)
t0 = time.perf_counter()
for wmax in (0, 1, 10, 100):
    times = random_graph(rng, 100, 6000, wmax)
    k = rng.randint(1, 100)
    res = {name: fn(times, 100, k) for name, fn in ALL.items()}
    assert len(set(res.values())) == 1, (wmax, res)
    print(f"n=100, E=6000, w <= {wmax:3d}: đáp án {res['solution']} (mọi cách khớp)")

# đồ thị thưa nhưng liên thông mạnh: vòng tròn 1->2->...->100->1 với trọng số 100
ring = [[i, i % 100 + 1, 100] for i in range(1, 101)]
res = {name: fn(ring, 100, 1) for name, fn in ALL.items()}
assert set(res.values()) == {99 * 100}, res
print("vòng tròn 100 nút, w = 100, k = 1:", res["solution"])

# đồ thị đầy đủ có hướng gần tối đa nhưng k không có cạnh đi ra
no_out = [[u, v, 1] for u in range(2, 101) for v in range(1, 101) if u != v][:6000]
assert set(fn(no_out, 100, 1) for fn in ALL.values()) == {-1}
print("k không có cạnh đi ra -> -1 với mọi cách")
print("thời gian:", round(time.perf_counter() - t0, 2), "s")

n=100, E=6000, w <=   0: đáp án 0 (mọi cách khớp)
n=100, E=6000, w <=   1: đáp án 0 (mọi cách khớp)


n=100, E=6000, w <=  10: đáp án 0 (mọi cách khớp)
n=100, E=6000, w <= 100: đáp án 11 (mọi cách khớp)


vòng tròn 100 nút, w = 100, k = 1: 9900
k không có cạnh đi ra -> -1 với mọi cách
thời gian: 0.25 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy **tổng** các khoảng cách thay vì `max`

In [21]:
def wrong_sum(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heappush(heap, (d + w, v))
    total = sum(dist[1:])                      # SAI: tín hiệu chạy song song
    return -1 if total == inf else total


ex1 = ([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2)
print("sai :", wrong_sum(*ex1), "| đúng:", delay_dijkstra_heap(*ex1))
assert wrong_sum(*ex1) == 4 and delay_dijkstra_heap(*ex1) == 2

sai : 4 | đúng: 2


**Vì sao sai:** các nhánh chạy **đồng thời**; thời điểm cả mạng nhận xong là thời điểm của nút **muộn nhất**, tức `max` chứ không phải `sum`. **Cách đúng:** `max(dist[1:])`.

### Lỗi 2 — Quên nút không tới được (chỉ lấy `max` trong các khoảng cách hữu hạn)

In [22]:
def wrong_ignore_unreachable(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heappush(heap, (d + w, v))
    return max(x for x in dist[1:] if x != inf)   # SAI: bỏ qua các nút còn inf


print("sai :", wrong_ignore_unreachable([[1, 2, 1]], 2, 2), "| đúng:", delay_dijkstra_heap([[1, 2, 1]], 2, 2))
assert wrong_ignore_unreachable([[1, 2, 1]], 2, 2) == 0 and delay_dijkstra_heap([[1, 2, 1]], 2, 2) == -1

sai : 0 | đúng: -1


**Vì sao sai:** đề yêu cầu **tất cả `n` nút** nhận được tín hiệu; nếu có nút còn `inf` thì không thể → `-1`. **Cách đúng:** lấy `max` **cả những `inf`**, rồi nếu `== inf` trả `-1`.

### Lỗi 3 — Nhầm chỉ số: nút `1..n` nhưng mảng chỉ có `n` phần tử

In [23]:
def wrong_index(times, n, k):
    dist = [inf] * n                           # SAI: cần n + 1 phần tử (hoặc trừ 1 mọi chỗ)
    dist[k] = 0                                # k = n sẽ vượt biên
    return dist


try:
    wrong_index([[1, 2, 1]], 2, 2)
except IndexError as e:
    print("IndexError khi k = n:", e)

# Python còn có bẫy chỉ số 0 / chỉ số âm: dist[0] là phần tử của 'nút 0' không tồn tại
dist = [inf, 3, 7, 5]
print("max(dist) =", max(dist), "-> bị inf của nút 0 phá hỏng; phải dùng max(dist[1:]) =", max(dist[1:]))
assert max(dist) == inf and max(dist[1:]) == 7

IndexError khi k = n: list assignment index out of range
max(dist) = inf -> bị inf của nút 0 phá hỏng; phải dùng max(dist[1:]) = 7


**Vì sao sai:** đề đánh số nút `1..n`; một mảng độ dài `n` chỉ có chỉ số `0..n-1`. **Hai cách đúng:** (a) dùng mảng `n + 1` phần tử, bỏ phần tử `0` khi lấy `max(dist[1:])`; (b) trừ 1 ở mọi chỗ (`k - 1`, `u - 1`, `v - 1`). Đừng trộn hai cách.

### Lỗi 4 — Đánh dấu "đã thăm" lúc **đẩy vào** heap (thay vì lúc lấy ra)

In [24]:
def wrong_visit_on_push(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    seen = {k}
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        for v, w in g[u]:
            if v not in seen:                  # SAI: lần đầu thấy v chưa chắc là đường ngắn nhất
                seen.add(v)
                dist[v] = d + w
                heappush(heap, (d + w, v))
    ans = max(dist[1:])
    return -1 if ans == inf else ans


tricky = ([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1)
print("sai :", wrong_visit_on_push(*tricky), "| đúng:", delay_dijkstra_heap(*tricky))
assert wrong_visit_on_push(*tricky) == 5 and delay_dijkstra_heap(*tricky) == 2

sai : 5 | đúng: 2


**Vì sao sai:** từ `1` ta thấy `2` đầu tiên qua cạnh `5`, nhưng đường qua `3` (`1 + 1 = 2`) ngắn hơn. Đánh dấu thăm khi đẩy vào khiến ta **không bao giờ cập nhật lại** `dist[2]`. Phải đánh dấu "chốt" khi **lấy ra** (hoặc dùng điều kiện `nd < dist[v]` để cho phép cải thiện nhiều lần và bỏ bản ghi cũ).

### Lỗi 5 — Dùng BFS thường (bỏ qua trọng số)

In [25]:
def wrong_bfs(times, n, k):
    g = build_graph(times, n)
    level = {k: 0}
    q = deque([k])
    while q:
        u = q.popleft()
        for v, w in g[u]:
            if v not in level:                 # SAI: đếm số cạnh, không cộng trọng số
                level[v] = level[u] + 1
                q.append(v)
    if len(level) < n:
        return -1
    return max(level.values())


print("sai :", wrong_bfs(*tricky), "| đúng:", delay_dijkstra_heap(*tricky))
assert wrong_bfs(*tricky) == 1 and delay_dijkstra_heap(*tricky) == 2

sai : 1 | đúng: 2


**Vì sao sai:** BFS cho đường ngắn nhất theo **số cạnh**, chỉ đúng khi mọi trọng số bằng nhau. Ở đây hai cạnh `1→3 (1)` rồi `3→2 (1)` tốn 2 thời gian nhưng 2 cạnh; còn BFS thấy `2` ở "tầng 1" qua cạnh trọng số 5 và trả `1`. **Cách đúng:** Dijkstra (hoặc Bellman–Ford) với trọng số thật.

### Lỗi 6 — Coi cạnh là **vô hướng**

In [26]:
def wrong_undirected(times, n, k):
    g = [[] for _ in range(n + 1)]
    for u, v, w in times:
        g[u].append((v, w))
        g[v].append((u, w))                    # SAI: cạnh chỉ đi một chiều
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heappush(heap, (d + w, v))
    ans = max(dist[1:])
    return -1 if ans == inf else ans


print("sai :", wrong_undirected([[1, 2, 1]], 2, 2), "| đúng:", delay_dijkstra_heap([[1, 2, 1]], 2, 2))
assert wrong_undirected([[1, 2, 1]], 2, 2) == 1 and delay_dijkstra_heap([[1, 2, 1]], 2, 2) == -1

sai : 1 | đúng: -1


**Vì sao sai:** đề nói cạnh `(u, v, w)` là tín hiệu đi từ `u` tới `v`, **không** ngược lại. **Cách đúng:** chỉ thêm `g[u].append((v, w))`.

### Lỗi 7 (bonus) — Bỏ `if d > dist[u]: continue`

Bỏ dòng kiểm tra bản ghi cũ **không làm sai đáp án** (vì nới lỏng vẫn đúng) nhưng làm ta mở rộng cùng một nút nhiều lần: chậm hơn. Ta đếm số lần mở rộng để thấy:

In [27]:
def dijkstra_count_expansions(times, n, k, skip_stale):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    expansions = 0
    while heap:
        d, u = heappop(heap)
        if skip_stale and d > dist[u]:
            continue
        expansions += 1
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heappush(heap, (d + w, v))
    return expansions, dist[1:]


rng = random.Random(7)
times = random_graph(rng, 100, 6000, 100)
e1, d1 = dijkstra_count_expansions(times, 100, 1, True)
e2, d2 = dijkstra_count_expansions(times, 100, 1, False)
print("có bỏ bản ghi cũ:", e1, "lần mở rộng | không bỏ:", e2, "lần | cùng kết quả:", d1 == d2)
assert d1 == d2 and e1 <= 100 and e2 >= e1

có bỏ bản ghi cũ: 100 lần mở rộng | không bỏ: 368 lần | cùng kết quả: True


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "đường ngắn nhất từ một nguồn":**

- Đồ thị có hướng, có trọng số **không âm**, hỏi "thời gian/khoảng cách tối thiểu tới ..." → **Dijkstra**.
- Trọng số **bằng nhau** (hoặc cạnh không trọng số) → BFS là đủ.
- Trọng số có thể **âm** → Bellman–Ford (hoặc SPFA với cảnh giác chu trình âm).
- Cần khoảng cách **mọi cặp** với `n` nhỏ → Floyd–Warshall.
- "Thời gian để TẤT CẢ nhận được" → `max` của các khoảng cách ngắn nhất, và kiểm tra `inf`.

**Mẹo nhớ:**

1. Khung Dijkstra heap: `dist = [inf]*(n+1)`; `heap = [(0, k)]`; lấy ra, bỏ bản ghi cũ, nới lỏng, đẩy vào.
2. Phần tử heap là `(khoảng cách, nút)` — khoảng cách đứng **trước** để heap xếp đúng.
3. Nút đánh số `1..n` → dùng mảng `n + 1` và `max(dist[1:])`.
4. `inf` sau cùng → `-1`.
5. Luôn nghĩ tới trọng số `0`: Dijkstra vẫn đúng.
6. Họ hàng: LeetCode 1514 (Path with Maximum Probability), 787 (Cheapest Flights Within K Stops), 1631 (Path With Minimum Effort), 1334, 2092.

## 15. Biến thể

### 15.1. Dựng lại **đường đi** tới nút xa nhất

Thêm mảng `prev[v]` ghi nút đứng trước `v` trên đường ngắn nhất; sau đó lần ngược từ nút xa nhất. Kiểm chứng: tổng trọng số dọc đường bằng `dist`.

In [28]:
def dijkstra_with_path(times, n, k):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    prev = [None] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                prev[v] = u
                heappush(heap, (d + w, v))
    return dist, prev


def path_to(prev, k, target):
    path = [target]
    while path[-1] != k:
        path.append(prev[path[-1]])
    return path[::-1]


times, n, k = [[1, 2, 5], [1, 3, 1], [3, 2, 1], [2, 4, 1]], 4, 1
dist, prev = dijkstra_with_path(times, n, k)
far = max(range(1, n + 1), key=lambda x: dist[x])
p = path_to(prev, k, far)
w = {(u, v): c for u, v, c in times}
print("nút xa nhất:", far, "| đường đi:", p, "| tổng trọng số:", sum(w[(a, b)] for a, b in zip(p, p[1:])), "| dist:", dist[far])
assert p == [1, 3, 2, 4] and sum(w[(a, b)] for a, b in zip(p, p[1:])) == dist[far] == 3

nút xa nhất: 4 | đường đi: [1, 3, 2, 4] | tổng trọng số: 3 | dist: 3


### 15.2. Thuật toán Dial (xô theo khoảng cách) cho trọng số nguyên nhỏ

Vì trọng số là số nguyên `0..100` và khoảng cách tối đa `<= 99 * 100 = 9900`, ta có thể thay heap bằng một mảng **xô** (bucket): `bucket[d]` chứa các nút có khoảng cách tạm `d`; duyệt `d` tăng dần. Thời gian `O(E + W·n)` với `W` là trọng số lớn nhất.

In [29]:
def delay_dial(times, n, k):
    g = build_graph(times, n)
    maxd = 100 * n                                   # nd tạm thời có thể tới 100*(n-1) + 100
    dist = [inf] * (n + 1)
    dist[k] = 0
    bucket = [[] for _ in range(maxd + 1)]
    bucket[0].append(k)
    for d in range(maxd + 1):
        i = 0
        while i < len(bucket[d]):                  # bucket[d] có thể được thêm khi trọng số = 0
            u = bucket[d][i]
            i += 1
            if dist[u] != d:
                continue
            for v, w in g[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    bucket[nd].append(v)
    ans = max(dist[1:])
    return -1 if ans == inf else ans


rng = random.Random(152)
for _ in range(300):
    n = rng.randint(2, 25)
    times = random_graph(rng, n, rng.randint(1, min(80, n * (n - 1))), rng.choice([0, 3, 100]))
    k = rng.randint(1, n)
    assert delay_dial(times, n, k) == delay_dijkstra_heap(times, n, k)
assert delay_dial(chain100, 100, 1) == 9900
print("15.2 OK")

15.2 OK


### 15.3. Chỉ tới **một đích** với **tối đa `m` cạnh** (kiểu LeetCode 787)

Bellman–Ford theo **lớp**: sau mỗi vòng ta chỉ dùng giá trị của vòng trước (sao chép mảng) để mỗi vòng thêm đúng một cạnh. Kiểm chứng bằng oracle đường đơn có giới hạn số cạnh (với trọng số không âm, đường đi tối ưu có thể chọn là đường đơn).

In [30]:
def min_time_at_most_edges(times, n, k, target, m):
    dist = [inf] * (n + 1)
    dist[k] = 0
    for _ in range(m):
        new = dist[:]
        for u, v, w in times:
            if dist[u] + w < new[v]:
                new[v] = dist[u] + w
        dist = new
    return -1 if dist[target] == inf else dist[target]


times = [[1, 2, 5], [1, 3, 1], [3, 2, 1]]
print([min_time_at_most_edges(times, 3, 1, 2, m) for m in range(4)])
assert [min_time_at_most_edges(times, 3, 1, 2, m) for m in range(4)] == [-1, 5, 2, 2]
rng = random.Random(153)
for _ in range(300):
    n = rng.randint(2, 6)
    times = random_graph(rng, n, rng.randint(1, min(12, n * (n - 1))), 6)
    k, target, m = rng.randint(1, n), rng.randint(1, n), rng.randint(0, 5)
    want = oracle_dists(times, n, k, max_edges=m)[target]
    assert min_time_at_most_edges(times, n, k, target, m) == (-1 if want == inf else want)
print("15.3 OK")

[-1, 5, 2, 2]
15.3 OK


### 15.4. Nhiều nguồn cùng gửi tín hiệu

Nếu tín hiệu gửi **đồng thời** từ một tập nút, đặt `dist = 0` cho tất cả nguồn và đẩy hết vào heap. Cách tương đương: thêm một "siêu nguồn" `0` nối tới các nguồn bằng cạnh trọng số `0`. Kiểm chứng hai cách cho cùng kết quả.

In [31]:
def delay_multi(times, n, sources):
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    heap = []
    for s in sources:
        dist[s] = 0
        heappush(heap, (0, s))
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heappush(heap, (d + w, v))
    ans = max(dist[1:])
    return -1 if ans == inf else ans


def delay_multi_super(times, n, sources):
    # nút n + 1 là siêu nguồn nối tới mọi nguồn bằng cạnh 0; nút ấy tự nó đạt thời điểm 0
    aug = times + [[n + 1, s, 0] for s in sources]
    return delay_dijkstra_heap(aug, n + 1, n + 1)


rng = random.Random(154)
for _ in range(300):
    n = rng.randint(2, 15)
    times = random_graph(rng, n, rng.randint(1, min(40, n * (n - 1))), 9)
    sources = rng.sample(range(1, n + 1), rng.randint(1, min(3, n)))
    assert delay_multi(times, n, sources) == delay_multi_super(times, n, sources)
print("15.4 OK:", delay_multi([[1, 2, 4], [3, 2, 1]], 3, [1, 3]))

15.4 OK: 1


### 15.5. Chọn nguồn **tốt nhất** (thời gian phủ toàn mạng nhỏ nhất)

Thử mọi `k` và chọn `k` cho đáp án nhỏ nhất (bỏ qua các `k` trả `-1`). Với `n <= 100`: `n` lần Dijkstra. Đối chiếu với Floyd–Warshall: `max` của hàng `k`.

In [32]:
def best_source(times, n):
    best = (inf, None)
    for k in range(1, n + 1):
        t = delay_dijkstra_heap(times, n, k)
        if t != -1 and t < best[0]:
            best = (t, k)
    return best if best[1] is not None else (-1, None)


def best_source_floyd(times, n):
    d = [[inf] * (n + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        d[i][i] = 0
    for u, v, w in times:
        d[u][v] = min(d[u][v], w)
    for t in range(1, n + 1):
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                d[i][j] = min(d[i][j], d[i][t] + d[t][j])
    best = (inf, None)
    for k in range(1, n + 1):
        t = max(d[k][1:])
        if t < best[0]:
            best = (t, k)
    return best if best[1] is not None else (-1, None)


rng = random.Random(155)
for _ in range(200):
    n = rng.randint(2, 8)
    times = random_graph(rng, n, rng.randint(1, min(30, n * (n - 1))), 9)
    a, b = best_source(times, n), best_source_floyd(times, n)
    assert a[0] == b[0], (times, a, b)         # thời gian bằng nhau (nguồn có thể hòa)
print("15.5 OK:", best_source([[1, 2, 1], [2, 3, 1], [3, 1, 1]], 3))

15.5 OK: (2, 1)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bellman–Ford | nới lỏng mọi cạnh `n-1` vòng | `O(n·E)` | `O(n)` | trọng số âm, `n·E` nhỏ |
| 2. Floyd–Warshall | DP qua nút trung gian | `O(n^3)` | `O(n^2)` | cần mọi cặp, `n` rất nhỏ |
| 3. Dijkstra mảng | quét tìm nút nhỏ nhất | `O(n^2)` | `O(n^2)` | đồ thị dày, `n` nhỏ |
| 4. Dijkstra + heap | min-heap `(dist, nút)` | `O(E log n)` | `O(n + E)` | **cách nộp**, đồ thị thưa/vừa |
| 5. SPFA | hàng đợi các nút vừa cải thiện | `O(E)` trung bình | `O(n + E)` | nhanh thực tế, tránh khi cần đảm bảo xấu nhất |

Khuyến nghị: nộp **cách 4**; dùng Bellman–Ford/Floyd/oracle DFS để tự kiểm thử.

In [33]:
rng = random.Random(16)
for _ in range(400):
    n = rng.randint(2, 6)
    times = random_graph(rng, n, rng.randint(1, min(14, n * (n - 1))), 8)
    k = rng.randint(1, n)
    res = {name: fn(times, n, k) for name, fn in ALL.items()}
    res["oracle"] = oracle_delay(times, n, k)
    res["dial"] = delay_dial(times, n, k)
    assert len(set(res.values())) == 1, (times, n, k, res)
print("400 đồ thị: 8 hàm cho cùng đáp án")

400 đồ thị: 8 hàm cho cùng đáp án


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(build_graph([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4)[2])
print(delay_dijkstra_heap([[1, 2, 1]], 2, 2))
print(delay_dijkstra_heap([[1, 2, 1], [2, 3, 2]], 3, 1))
```

In [34]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay Dijkstra (dễ-vừa)

Với `times = [[1,2,4],[1,3,1],[3,2,2],[2,4,1]]`, `n = 4`, `k = 1`: (a) khoảng cách ngắn nhất `dist[1..4]`; (b) thứ tự các nút được **chốt**; (c) đáp án cuối cùng.

In [35]:
# Bài 2: điền kết quả tính tay
hand_dist = None         # list [dist1, dist2, dist3, dist4]
hand_order = None        # list thứ tự chốt, ví dụ [1, 3, ...]
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành Dijkstra với heap.

In [36]:
def exercise_3_delay(times: List[List[int]], n: int, k: int) -> int:
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    # TODO 1: lặp khi heap còn phần tử
    #     TODO 2: d, u = heappop(heap)
    #     TODO 3: nếu d > dist[u] thì bỏ qua (bản ghi cũ)
    #     TODO 4: với mỗi (v, w) trong g[u]: nếu d + w < dist[v] thì cập nhật dist[v] và heappush (d + w, v)
    # TODO 5: tính ans = max(dist[1:]) rồi trả -1 nếu ans == inf, ngược lại trả ans
    return None


# assert exercise_3_delay([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
# assert exercise_3_delay([[1, 2, 1]], 2, 2) == -1
# assert exercise_3_delay([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1) == 2

### Bài 4 — Tự viết lại (vừa-khó): Bellman–Ford

Không nhìn lại mục 5, hãy tự viết `exercise_4_bf(times, n, k)` bằng Bellman–Ford (lặp `n - 1` vòng nới lỏng mọi cạnh).

```python
exercise_4_bf([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) -> 2
exercise_4_bf([[1, 2, 1]], 2, 2) -> -1
exercise_4_bf([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1) -> 2
```

In [37]:
def exercise_4_bf(times: List[List[int]], n: int, k: int) -> int:
    # TODO
    pass


# assert exercise_4_bf([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
# assert exercise_4_bf([[1, 2, 1]], 2, 2) == -1
# assert exercise_4_bf([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1) == 2

### Bài 5 — Mini challenge: tối đa `m` cạnh trên mỗi đường (khó)

Mỗi tín hiệu chỉ được đi qua **tối đa `m` cạnh**. Viết `exercise_5_at_most_edges(times, n, k, m)` trả về thời gian nhỏ nhất để **cả `n` nút** nhận được tín hiệu khi mỗi đường đi dùng `<= m` cạnh, hoặc `-1` nếu có nút không thể nhận.

```python
t = [[1, 2, 5], [1, 3, 1], [3, 2, 1]]
exercise_5_at_most_edges(t, 3, 1, 0) -> -1
exercise_5_at_most_edges(t, 3, 1, 1) -> 5
exercise_5_at_most_edges(t, 3, 1, 2) -> 2
```

Gợi ý hướng nghĩ: Bellman–Ford theo lớp (sao chép mảng mỗi vòng), chạy đúng `m` vòng.

In [38]:
def exercise_5_at_most_edges(times: List[List[int]], n: int, k: int, m: int) -> int:
    # TODO
    pass


# t = [[1, 2, 5], [1, 3, 1], [3, 2, 1]]
# assert exercise_5_at_most_edges(t, 3, 1, 0) == -1
# assert exercise_5_at_most_edges(t, 3, 1, 1) == 5
# assert exercise_5_at_most_edges(t, 3, 1, 2) == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `g[2]` là danh sách các `(đích, trọng số)` đi ra từ nút `2`.
- Mức 2: với `k = 2` và chỉ có cạnh `1 → 2`, nút `1` không tới được. Ở dòng 3: `1 → 2` mất 1, `2 → 3` mất 2.
- Mức 3: dòng 1 là `[(1, 1), (3, 1)]`; dòng 2 là `-1`; dòng 3 là `3` (`1 + 2`).

**Bài 2**
- Mức 1: bắt đầu `dist = [0, inf, inf, inf]`; từ `1` cập nhật `2` và `3`.
- Mức 2: `1 → 2` mất 4, `1 → 3` mất 1; chốt `3` (nhỏ nhất) rồi `3 → 2` mất `1 + 2 = 3 < 4`, cập nhật `dist[2] = 3`.
- Mức 3: (a) `[0, 3, 1, 4]`; (b) `[1, 3, 2, 4]`; (c) `4` (nút `4` tới qua `2`: `3 + 1`).

**Bài 3**
- Mức 1: giống mục 7.3.
- Mức 2: `while heap:`; `d, u = heappop(heap)`; `if d > dist[u]: continue`.
- Mức 3: trong vòng `for v, w in g[u]` dùng `nd = d + w`; cuối hàm `ans = max(dist[1:])`.

**Bài 4**
- Mức 1: `dist = [inf]*(n+1)`; `dist[k] = 0`.
- Mức 2: `for _ in range(n - 1):` rồi `for u, v, w in times:` nếu `dist[u] + w < dist[v]` thì cập nhật.
- Mức 3: cuối cùng `max(dist[1:])`, trả `-1` nếu bằng `inf`.

**Bài 5**
- Mức 1: mỗi vòng chỉ được thêm **một** cạnh vào mỗi đường.
- Mức 2: trong vòng, đọc từ mảng cũ `dist` nhưng ghi vào mảng mới `new = dist[:]`; cuối vòng `dist = new`.
- Mức 3: lặp đúng `m` vòng; kết quả `max(dist[1:])`, `-1` nếu còn `inf` (nên `m = 0` cho `-1` khi `n > 1`).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [39]:
print(build_graph([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4)[2])
print(delay_dijkstra_heap([[1, 2, 1]], 2, 2))
print(delay_dijkstra_heap([[1, 2, 1], [2, 3, 2]], 3, 1))

assert build_graph([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4)[2] == [(1, 1), (3, 1)]
assert delay_dijkstra_heap([[1, 2, 1]], 2, 2) == -1
assert delay_dijkstra_heap([[1, 2, 1], [2, 3, 2]], 3, 1) == 3

[(1, 1), (3, 1)]
-1
3


### Đáp án Bài 2

| bước | chốt `(d, u)` | nới lỏng | `dist[1..4]` |
|---|---|---|---|
| bắt đầu | - | - | `[0, inf, inf, inf]` |
| 1 | `(0, 1)` | `2: inf→4`, `3: inf→1` | `[0, 4, 1, inf]` |
| 2 | `(1, 3)` | `2: 4→3` (qua `3`) | `[0, 3, 1, inf]` |
| 3 | `(3, 2)` | `4: inf→4` | `[0, 3, 1, 4]` |
| 4 | `(4, 2)` | **bản ghi cũ** (`4 > 3`), bỏ qua | `[0, 3, 1, 4]` |
| 5 | `(4, 4)` | không có cạnh đi ra | `[0, 3, 1, 4]` |

Thứ tự chốt `[1, 3, 2, 4]`, đáp án `4`.

In [40]:
t2 = ([[1, 2, 4], [1, 3, 1], [3, 2, 2], [2, 4, 1]], 4, 1)
res = trace_dijkstra(*t2)
assert res == 4
assert show_dist(*t2) == [0, 3, 1, 4]
assert oracle_delay(*t2) == 4
print("Bài 2 OK")

bắt đầu: dist = [0, inf, inf, inf], heap = [(0, 1)]
bước 1: lấy (0,1) chốt nút 1; nới lỏng [2: inf->4; 3: inf->1] | dist = [0, 4, 1, inf] | heap = [(1, 3), (4, 2)]
bước 2: lấy (1,3) chốt nút 3; nới lỏng [2: 4->3] | dist = [0, 3, 1, inf] | heap = [(3, 2), (4, 2)]
bước 3: lấy (3,2) chốt nút 2; nới lỏng [4: inf->4] | dist = [0, 3, 1, 4] | heap = [(4, 2), (4, 4)]
bước 4: lấy (4,2) -> BẢN GHI CŨ (dist[2] = 3), bỏ qua
bước 5: lấy (4,4) chốt nút 4; nới lỏng [không có cạnh đi ra] | dist = [0, 3, 1, 4] | heap = []
max(dist) = 4 -> 4
Bài 2 OK


### Đáp án Bài 3

In [41]:
def exercise_3_delay_solution(times: List[List[int]], n: int, k: int) -> int:
    g = build_graph(times, n)
    dist = [inf] * (n + 1)
    dist[k] = 0
    heap = [(0, k)]
    while heap:
        d, u = heappop(heap)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heappush(heap, (nd, v))
    ans = max(dist[1:])
    return -1 if ans == inf else ans


assert exercise_3_delay_solution([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert exercise_3_delay_solution([[1, 2, 1]], 2, 2) == -1
assert exercise_3_delay_solution([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1) == 2
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(2, 6)
    times = random_graph(rng, n, rng.randint(1, min(12, n * (n - 1))), 6)
    k = rng.randint(1, n)
    assert exercise_3_delay_solution(times, n, k) == oracle_delay(times, n, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_bf_solution(times: List[List[int]], n: int, k: int) -> int:
    dist = [inf] * (n + 1)
    dist[k] = 0
    for _ in range(n - 1):
        for u, v, w in times:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    ans = max(dist[1:])
    return -1 if ans == inf else ans


assert exercise_4_bf_solution([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert exercise_4_bf_solution([[1, 2, 1]], 2, 2) == -1
assert exercise_4_bf_solution([[1, 2, 5], [1, 3, 1], [3, 2, 1]], 3, 1) == 2
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 6)
    times = random_graph(rng, n, rng.randint(1, min(12, n * (n - 1))), 6)
    k = rng.randint(1, n)
    assert exercise_4_bf_solution(times, n, k) == oracle_delay(times, n, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
def exercise_5_at_most_edges_solution(times: List[List[int]], n: int, k: int, m: int) -> int:
    dist = [inf] * (n + 1)
    dist[k] = 0
    for _ in range(m):
        new = dist[:]
        for u, v, w in times:
            if dist[u] + w < new[v]:
                new[v] = dist[u] + w
        dist = new
    ans = max(dist[1:])
    return -1 if ans == inf else ans


t5 = [[1, 2, 5], [1, 3, 1], [3, 2, 1]]
assert exercise_5_at_most_edges_solution(t5, 3, 1, 0) == -1
assert exercise_5_at_most_edges_solution(t5, 3, 1, 1) == 5
assert exercise_5_at_most_edges_solution(t5, 3, 1, 2) == 2
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(2, 6)
    times = random_graph(rng, n, rng.randint(1, min(12, n * (n - 1))), 6)
    k, m = rng.randint(1, n), rng.randint(0, 5)
    best = oracle_dists(times, n, k, max_edges=m)[1:]
    want = max(best)
    assert exercise_5_at_most_edges_solution(times, n, k, m) == (-1 if want == inf else want)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from heapq import heappush, heappop
from typing import List

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        inf = float("inf")
        g = [[] for _ in range(n + 1)]              # nút 1..n, bỏ phần tử 0
        for u, v, w in times:
            g[u].append((v, w))                     # cạnh CÓ HƯỚNG
        dist = [inf] * (n + 1)
        dist[k] = 0
        heap = [(0, k)]                             # (khoảng cách, nút)
        while heap:
            d, u = heappop(heap)
            if d > dist[u]:                         # bản ghi cũ
                continue
            for v, w in g[u]:
                if d + w < dist[v]:
                    dist[v] = d + w
                    heappush(heap, (dist[v], v))
        ans = max(dist[1:])                         # max, không phải sum
        return -1 if ans == inf else ans
```

Câu thần chú: **"Dijkstra từ `k`; đáp án là `max(dist[1:])`; còn `inf` thì `-1`."**

### Checklist trước khi submit

- [ ] Cạnh **có hướng**: chỉ `g[u].append((v, w))`.
- [ ] Mảng `dist` có `n + 1` phần tử; lấy `max(dist[1:])` (bỏ nút `0`).
- [ ] Đáp án là `max` (không phải tổng); nếu `== inf` thì trả `-1`.
- [ ] Phần tử heap là `(khoảng cách, nút)`; có dòng bỏ bản ghi cũ `if d > dist[u]: continue`.
- [ ] Không đánh dấu "đã thăm" lúc đẩy vào heap.
- [ ] Trọng số `0` vẫn đúng; Dijkstra không dùng được nếu có trọng số âm (ở bài này không có).
- [ ] Test: nút không tới được, `k` không có cạnh đi ra, chu trình, đường tắt, dây chuyền 100 nút (`9900`).
- [ ] Skeleton đúng: `class Solution`, method `networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int`; độ phức tạp `O(E log n)`.